# Quantitative Width Selection

Evaluating circle widths (2, 4, 6, 8) on a subset of VOC (100 images).

Metrics:
- Mean GT-Concept Rank after Sum aggregation
- Mean GT-Concept Activation Strength
- Mean Sparsity (non-zero concepts per image)

In [ ]:
import sys
sys.path.insert(0, "/workspaces/Object-Centric-Concept-Bottlenecks")

import torch
import open_clip
import splice
from splice.model import SPLICE
from PIL import Image, ImageDraw
import torchvision.transforms.functional as TF
import torchvision.transforms.v2 as T
from torchvision.models.detection import maskrcnn_resnet50_fpn
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import os
import numpy as np
from tqdm import tqdm

from utils import draw_circle_on_image, generate_object_propsals

WIDTHS = [2, 4, 6, 8]
SUBSET_SIZE = 100

print("Imports OK")

In [ ]:
# MaskRCNN
maskrcnn = maskrcnn_resnet50_fpn(pretrained=True)
maskrcnn.eval()
maskrcnn = maskrcnn.to("cuda")
print("MaskRCNN loaded")

# SpLiCE
concepts = torch.load("/root/.cache/splice/embeddings/open_clip:ViT-B-32_laion_10000_embeddings.pt", map_location="cuda")
image_mean = torch.load("/root/.cache/splice/means/open_clip_ViT-B-32_image.pt", map_location="cuda")
clip_backbone, _, splice_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_backbone.eval()
clip_backbone = clip_backbone.to("cuda")
splice_model = SPLICE(image_mean=image_mean, dictionary=concepts, clip=clip_backbone, device="cuda", l1_penalty=0.25, return_weights=True)
splice_model.eval()
vocab = splice.get_vocabulary("laion", vocabulary_size=10000)
print("SpLiCE loaded")

In [ ]:
VOC_IMG = "/workspaces/Object-Centric-Concept-Bottlenecks/data/voc/VOCdevkit/VOC2012/JPEGImages/"
VOC_ANN = "/workspaces/Object-Centric-Concept-Bottlenecks/data/voc/VOCdevkit/VOC2012/Annotations/"

to_tensor = T.Compose([T.Resize((640, 640)), T.ToTensor()])

def load_voc_sample(filename):
    pil = Image.open(VOC_IMG + filename).convert("RGB")
    tensor = to_tensor(pil)
    
    tree = ET.parse(VOC_ANN + filename.replace(".jpg", ".xml"))
    root = tree.getroot()
    labels = list(set([obj.find("name").text for obj in root.findall("object")]))
    return pil, tensor, labels

# Pick 100 images that have annotations
all_files = sorted(os.listdir(VOC_IMG))
subset = []
for fname in all_files:
    ann_path = VOC_ANN + fname.replace(".jpg", ".xml")
    if os.path.exists(ann_path):
        subset.append(fname)
    if len(subset) == SUBSET_SIZE:
        break

print(f"Loaded {len(subset)} annotated VOC images")
print(f"Example: {subset[0]} — {load_voc_sample(subset[0])[2]}")


In [ ]:
def get_splice_emb(image_pil):
    img_t = splice_preprocess(image_pil).unsqueeze(0).to("cuda")
    with torch.no_grad():
        emb, _ = splice_model.encode_image(img_t)
    return emb.squeeze(0)

def get_concept_rank(emb, vocab, gt_labels, top_k=20):
    top_idx = emb.argsort(descending=True)[:top_k]
    for rank, idx in enumerate(top_idx):
        word = vocab[idx]
        for label in gt_labels:
            if label in word or word in label:
                return rank + 1, word, emb[idx].item()
    return None, None, 0.0

def get_sparsity(emb):
    return int((emb > 0).sum().item())


## Main Evaluation Loop

For each image:
1. Run MaskRCNN to detect objects
2. For each width: encode all objects with Red Circle + full image
3. Apply Sum aggregation
4. Measure GT-Concept Rank, Activation Strength, Sparsity

In [ ]:
results = {w: {"ranks": [], "strengths": [], "sparsities": []} for w in WIDTHS}

for fname in tqdm(subset):
    try:
        pil, tensor, gt_labels = load_voc_sample(fname)
    except:
        continue

    # MaskRCNN
    with torch.no_grad():
        boxes_list, scores_list = generate_object_propsals(
            [tensor.to("cuda")], maskrcnn,
            min_size=0.02, max_size=0.85,
            iou_threshold=0.5, minimum_score=0.3,
        )
    boxes  = boxes_list[0]
    scores = scores_list[0]

    if len(boxes) == 0:
        continue

    boxes = boxes[:7]  # max 7 objects

    # Full image embedding
    full_emb = get_splice_emb(pil)

    for width in WIDTHS:
        obj_embs = torch.stack([
            get_splice_emb(draw_circle_on_image(tensor, box, width=width))
            for box in boxes
        ])
        sum_agg = full_emb + obj_embs.sum(dim=0)

        rank, _, strength = get_concept_rank(sum_agg, vocab, gt_labels)
        sparsity = get_sparsity(sum_agg)

        if rank is not None:
            results[width]["ranks"].append(rank)
            results[width]["strengths"].append(strength)
        results[width]["sparsities"].append(sparsity)

print("Done!")

In [ ]:
print(f"{'Width':<8} {'Mean GT-Rank':<16} {'Mean Strength':<16} {'Mean Sparsity':<16} {'GT Found %':<12}")
print("-" * 70)

for width in WIDTHS:
    ranks     = results[width]["ranks"]
    strengths = results[width]["strengths"]
    sparsities = results[width]["sparsities"]

    # Auf welchem Rang erscheint das GT-Label-Konzept im Durchschnitt nach Sum-Aggregation
    mean_rank     = np.mean(ranks)     if ranks     else float("nan")
    # Wie stark ist der Score des GT-Konzepts nach Sum-Aggregation
    mean_strength = np.mean(strengths) if strengths else float("nan")
    # Wie viele Konzepte werden durchschnittlich aktiviert (Non-zero)
    mean_sparsity = np.mean(sparsities)
    # Bei wie vielen Bildern wurde das GT-Label überhaupt in den Top-20 Konzepten gefunden
    found_pct     = len(ranks) / len(sparsities) * 100

    print(f"w={width:<5} {mean_rank:<16.2f} {mean_strength:<16.4f} {mean_sparsity:<16.1f} {found_pct:<12.1f}")


## Datasets: VOC Subset (1000) | COCO Subset (500)

In [ ]:
import json

# VOC Subset
voc_subset = [f for f in sorted(os.listdir(VOC_IMG))
              if os.path.exists(VOC_ANN + f.replace(".jpg", ".xml"))][:1000]
print(f"VOC Subset: {len(voc_subset)} images")

# COCO Subset
COCO_IMG = "/workspaces/Object-Centric-Concept-Bottlenecks/datasets/coco/images/val2017/"
COCO_ANN_PATH = "/workspaces/Object-Centric-Concept-Bottlenecks/datasets/coco/annotations/instances_val2017.json"

with open(COCO_ANN_PATH) as f:
    coco_data = json.load(f)

cat_id_to_name = {cat["id"]: cat["name"] for cat in coco_data["categories"]}
img_id_to_fname = {img["id"]: img["file_name"] for img in coco_data["images"]}
fname_to_cats = {}
for ann in coco_data["annotations"]:
    fname = img_id_to_fname[ann["image_id"]]
    fname_to_cats.setdefault(fname, set()).add(cat_id_to_name[ann["category_id"]])

coco_subset = [f for f in sorted(os.listdir(COCO_IMG)) if f in fname_to_cats][:500]
print(f"COCO Subset: {len(coco_subset)} images")

In [ ]:
def run_evaluation(file_list, load_fn, dataset_name):
    res = {w: {"ranks": [], "strengths": [], "sparsities": []} for w in WIDTHS}

    for fname in tqdm(file_list, desc=dataset_name):
        try:
            pil, tensor, gt_labels = load_fn(fname)
        except:
            continue

        with torch.no_grad():
            boxes_list, scores_list = generate_object_propsals(
                [tensor.to("cuda")], maskrcnn,
                min_size=0.02, max_size=0.85,
                iou_threshold=0.5, minimum_score=0.3,
            )
        boxes = boxes_list[0][:7]

        if len(boxes) == 0:
            continue

        full_emb = get_splice_emb(pil)

        for width in WIDTHS:
            obj_embs = torch.stack([
                get_splice_emb(draw_circle_on_image(tensor, box, width=width))
                for box in boxes
            ])
            sum_agg = full_emb + obj_embs.sum(dim=0)
            rank, _, strength = get_concept_rank(sum_agg, vocab, gt_labels)
            sparsity = get_sparsity(sum_agg)

            if rank is not None:
                res[width]["ranks"].append(rank)
                res[width]["strengths"].append(strength)
            res[width]["sparsities"].append(sparsity)

    return res

def print_results(res, dataset_name):
    print(f"\n{'='*70}")
    print(f"Dataset: {dataset_name}")
    print(f"{'Width':<8} {'Mean GT-Rank':<16} {'Mean Strength':<16} {'Mean Sparsity':<16} {'GT Found %':<12}")
    print("-" * 70)
    for width in WIDTHS:
        ranks     = res[width]["ranks"]
        strengths = res[width]["strengths"]
        sparsities = res[width]["sparsities"]
        mean_rank     = np.mean(ranks)     if ranks     else float("nan")
        mean_strength = np.mean(strengths) if strengths else float("nan")
        mean_sparsity = np.mean(sparsities)
        found_pct     = len(ranks) / len(sparsities) * 100
        print(f"w={width:<5} {mean_rank:<16.2f} {mean_strength:<16.4f} {mean_sparsity:<16.1f} {found_pct:<12.1f}")


In [ ]:
def load_coco_sample(filename):
    pil = Image.open(COCO_IMG + filename).convert("RGB")
    tensor = to_tensor(pil)
    gt_labels = list(fname_to_cats.get(filename, set()))
    return pil, tensor, gt_labels

In [ ]:
# VOC Full (~30-60 min auf GPU)
results_voc_subset = run_evaluation(voc_subset, load_voc_sample, "VOC Full")
print_results(results_voc_subset, "VOC Full")

In [ ]:
# COCO Subset (500 Bilder)
results_coco = run_evaluation(coco_subset, load_coco_sample, "COCO Subset (500)")
print_results(results_coco, "COCO Subset (500)")